# codesearch — сравнение би-энкодеров (GPU)

Прогоняет eval для всех конфигов из `configs/embedder-*.yaml` на одном и том же протоколе
(python + go, 1000 записей корпуса, по 200 запросов на язык, **без реранкера** — сравниваем чистый retrieval).

**Перед запуском:** Runtime → Change runtime type → **T4 GPU**, затем Runtime → Run all.

Ориентировочное время: ~25–40 минут (большая часть — скачивание моделей и Qodo-1.5B).

In [ ]:
!nvidia-smi

In [ ]:
!git clone -b feat/embedder-comparison https://github.com/PopovDanil/code-searching-engine.git
%cd code-searching-engine

In [ ]:
!pip install -q -U transformers sentence-transformers
!pip install -q faiss-cpu tree-sitter tree-sitter-python tree-sitter-java tree-sitter-javascript tree-sitter-go tree-sitter-ruby tree-sitter-php datasets typer pyyaml

In [ ]:
import gc
import glob
import sys

import torch

sys.path.insert(0, "src")
from config import CodeSearchConfig
from evaluation.evaluate import evaluate_on_codesearchnet

LANGUAGES = ["python", "go"]
MAX_DATASET_RECORDS = 1000   # делится между языками поровну
MAX_QUERIES = 200            # на язык

results = {}
for path in sorted(glob.glob("configs/embedder-*.yaml")):
    name = path.split("embedder-")[1].rsplit(".", 1)[0]
    print("\n" + "=" * 70)
    print(f"  {name}   ({path})")
    print("=" * 70)
    config = CodeSearchConfig.from_yaml(path)
    try:
        res = evaluate_on_codesearchnet(
            config,
            languages=LANGUAGES,
            max_dataset_records=MAX_DATASET_RECORDS,
            max_queries=MAX_QUERIES,
        )
        results[name] = res
    except Exception as exc:
        print(f"FAILED: {exc}")
        results[name] = {}
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# Итоговая таблица (overall по всем языкам) — готовая строка для experiments.md
METRICS = ["Recall@1", "Recall@5", "Recall@10", "MRR", "NDCG@10"]

print("| Embedder | " + " | ".join(METRICS) + " |")
print("|---" * (len(METRICS) + 1) + "|")
for name, res in results.items():
    overall = res.get("overall", {}) if res else {}
    if overall:
        row = " | ".join(f"{overall[m]:.4f}" for m in METRICS)
    else:
        row = " | ".join("—" for _ in METRICS)
    print(f"| {name} | {row} |")

print()
# Разбивка по языкам
for name, res in results.items():
    for lang, vals in (res or {}).items():
        if lang == "overall":
            continue
        print(f"{name:>22s} / {lang:<8s}  " + "  ".join(f"{m}={vals[m]:.4f}" for m in METRICS))

## Что дальше

1. Markdown-таблицу из ячейки выше — в `experiments.md` (строка «сравнение би-энкодеров»).
2. Победителя прогнать **с реранкером** (в его yaml поставить `enable_reranking: true`) и на большем корпусе.
3. Если код-специфичная модель ≥ Qwen3-0.6B — меняем дефолт в `example_config.yaml`.